# ML final project
date: 30.06.2025

last updated: 01.07.2025

---

## 🦠 Project Title:
**Predicting Airborne Disease Spread Using Urban Mobility and Environmental Factors**

---

## 🎯 Core Objective:
Train a **spatiotemporal deep learning model** — such as a Graph Neural Network (GNN) or Transformer — to forecast the likelihood of increased disease spread in urban areas by combining:

* **Human mobility patterns**
* **Air quality and weather data**
* **Historical outbreak data**

---

## 🔍 Scientific Questions:

1. *Can mobility + air quality explain spatial clusters of disease spread better than historical cases alone?*
2. *Which environmental features (e.g., PM2.5, humidity) correlate most strongly with surges?*
3. *Can we generalize prediction models across cities or countries?*

---

## 📦 Data Sources

### 1. **Mobility Data**

* **[OpenSky Network](https://opensky-network.org/datasets)**: Real-time and historical air traffic data (can proxy global mobility)
* **[Google Community Mobility Reports (GCMR)](https://www.google.com/covid19/mobility/)**: % change in visits to workplaces, transit stations, etc. per country/city over time.
* **[Apple Mobility Trends](https://covid19.apple.com/mobility)** (archived): Driving, walking, transit indices by city.

**Tip**: GCMR is super easy to use — it’s CSV-based, per region, and aligns nicely with time-series modeling.

---

### 2. **Air Quality and Weather**

* **[OpenAQ API](https://docs.openaq.org/)**: Real-time and historical global air quality data (PM2.5, ozone, NO2, etc.)
* **[ERA5 Climate Reanalysis (Copernicus)](https://cds.climate.copernicus.eu/)**: High-res weather reanalysis — includes temperature, humidity, wind
* **[NOAA ISD or GSOD](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily)**: Daily weather from meteorological stations

---

### 3. **Disease Case Data**

* **[CSSE COVID-19 Dataset (Johns Hopkins)](https://github.com/CSSEGISandData/COVID-19)**: Case counts and deaths per location (well-maintained)
* **[Our World in Data COVID-19](https://github.com/owid/covid-19-data/tree/master/public/data)**: Includes testing and healthcare policy metrics
* **[WHO FluNet](https://www.who.int/tools/flunet)**: Weekly flu case data per country

---

## 🧠 Model Ideas

### 🅰️ **Option 1: Temporal Graph Neural Network (GNN)**

* Nodes: Cities or regions
* Edges: Mobility flow (inter-city movement)
* Features: PM2.5, humidity, population, lagged case counts
* Labels: New cases next day/week

Use models like:

* `ST-GCN` (Spatiotemporal Graph ConvNet)
* `TGAT` (Temporal Graph Attention Network)
* `DCRNN` (Diffusion Convolutional RNN)

### 🅱️ **Option 2: Transformer over Feature Sequences**

If you skip graph structure:

* Use concatenated time-series features
* Train a Transformer or TCN to forecast disease growth
* Faster and lighter than GNN, easier to prototype

---

## 🛠️ Technical Scope Plan

| Phase | Task                                                | Time Est. |
| ----- | --------------------------------------------------- | --------- |
| 🧼 1  | Collect & clean data (3 cities or 1 country)        | 3 days    |
| 📊 2  | Feature engineering (lags, weather smoothing)       | 2 days    |
| 🧠 3  | Model prototyping (Transformer baseline)            | 3 days    |
| 📈 4  | Train & evaluate (MSE, R², AUC)                     | 2 days    |
| 📘 5  | Visualization + analysis (attention maps, saliency) | 2–3 days  |

---

## 📚 Background Reading & Literature

* **COVID-19 Modeling with Deep Learning**:
  [“DeepCOVIDNet: An Interpretable Deep Learning Model for Predictive Surveillance” (2020)](https://arxiv.org/abs/2004.07449)
  (uses mobility and demographics for COVID-19 prediction)

* **Graph Neural Networks for Epidemiology**:
  [“EpiGNN: Epidemic Forecasting with Graph Neural Networks” (2021)](https://arxiv.org/abs/2103.13238)

* **Spatiotemporal GNNs** (survey):
  [“A Comprehensive Survey on Graph Neural Networks” (2021, Section 5)](https://arxiv.org/abs/1901.00596)

* **OpenAQ API Docs**:
  [https://docs.openaq.org/](https://docs.openaq.org/)

* **Google Mobility Reports**:
  [https://www.google.com/covid19/mobility/](https://www.google.com/covid19/mobility/)

---

## 💡 Final Tips

* **Start small**: Pick 3 cities (e.g., New York, Paris, Tokyo)
* **Temporal scope**: 6–12 months of data is enough
* **Validate on surges**: Use known COVID waves as “ground truth”
* Use **lags** (e.g. pollution today → cases 5–10 days later)
* Try **feature importance (SHAP)** or **attention maps** for interpretability

---


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests


## Part 1: Collect and Clean Data

### 1) Using Google Community Mobility Reports (GCMR): https://www.google.com/covid19/mobility/

For Israel we have 974 days of data (15.02.2020 - 15.10.2022)

In [2]:
## Download and load GCMR
# gcmr_path = 'https://www.gstatic.com/covid19/mobility/'
gcmr_path = '/home/barbpt/Desktop/ML_course/Final_project/' # downloaded
## This file is too large!!
# df_gcmr = pd.read_csv(f'{gcmr_path}Global_Mobility_Report.csv', low_memory=True)

# Create an empty list to hold filtered chunks
## we are using chunks to only load relevant data, since the total data is too large
chunks = []

# Read in chunks and filter
for chunk in pd.read_csv(f'{gcmr_path}Global_Mobility_Report.csv', chunksize=100_000,low_memory=False):
    # Keep rows where country_region == 'United States' and sub_region_1 == 'New York'
    ny_chunk = chunk[(chunk['country_region'] == 'United States') & 
                     (chunk['sub_region_1'] == 'New York')]
    if not ny_chunk.empty:
        chunks.append(ny_chunk)

# Concatenate the filtered chunks
df_ny = pd.concat(chunks, ignore_index=True)

# print(df_ny.head())



# # Filter for New York
# df_nyc = df_gcmr[(df_gcmr['country_region'] == 'United States') &
#                  (df_gcmr['sub_region_1'] == 'New York') &
#                  (df_gcmr['sub_region_2'] == 'New York County')]
# # Filter for Israel, and optionally just Tel Aviv District
# df_il = gcmr[(gcmr['country_region'] == 'Israel') & 
#              (gcmr['sub_region_1'] == 'Tel Aviv District') & 
#              (gcmr['sub_region_2'].isnull())]


# Convert date and compute composite mobility index
## create daily mobility index: Mobility index as the average (mean) of percentage change from baseline
df_ny['date'] = pd.to_datetime(df_ny['date'])
mobility_cols = ['workplaces_percent_change_from_baseline',
                 'transit_stations_percent_change_from_baseline',
                 'grocery_and_pharmacy_percent_change_from_baseline',
                 'retail_and_recreation_percent_change_from_baseline']
df_ny['mobility_index'] = df_ny[mobility_cols].mean(axis=1)
df_mobility = df_ny[['date', 'mobility_index']].dropna()


In [3]:
df_mobility

,date,mobility_index
0,2020-02-15,-1.00
1,2020-02-16,1.00
2,2020-02-17,-13.50
3,2020-02-18,-5.50
4,2020-02-19,-2.75
...,...,...
60121,2022-10-11,10.00
60122,2022-10-12,-18.00
60123,2022-10-13,5.00
60124,2022-10-14,3.00


### 2) COVID/flu cases (from Johns Hopkins or RKI)
 - https://github.com/owid/covid-19-data
 - https://github.com/CSSEGISandData/COVID-19
 
Flu:
 - https://www.who.int/

In [13]:
# Daily NYC COVID case data from GitHub
url_cases = 'https://raw.githubusercontent.com/nychealth/coronavirus-data/master/trends/data-by-day.csv'
df_cases = pd.read_csv(url_cases)
# print(df_cases.columns)

df_cases['date'] = pd.to_datetime(df_cases['date_of_interest']).dt.date
df_cases = df_cases[['date', 'CASE_COUNT']].rename(columns={'CASE_COUNT': 'covid_cases'})


In [14]:
# Load OWID global COVID-19 data
url_covid = 'https://covid.ourworldindata.org/data/owid-covid-data.csv'
df_covid = pd.read_csv(url_covid, usecols=['location', 'date', 'new_cases_smoothed'])

# Filter for Israel
df_covid = df_covid[df_covid['location'] == 'New York'].copy()
df_covid['date'] = pd.to_datetime(df_covid['date']).dt.date
df_covid = df_covid.rename(columns={'new_cases_smoothed': 'covid_cases'})
df_covid = df_covid[['date', 'covid_cases']].dropna()
df_covid.head()


,date,covid_cases


In [15]:
df_covid

,date,covid_cases


In [16]:
df_cases

,date,covid_cases
0,2020-02-29,1
1,2020-03-01,0
2,2020-03-02,0
3,2020-03-03,1
4,2020-03-04,5
...,...,...
1937,2025-06-19,119
1938,2025-06-20,136
1939,2025-06-21,97
1940,2025-06-22,75


### 3) Air quality (from OpenAQ or Ministry of Health)
 - https://www.gov.il/he/departments/ministry_of_environmental_protection/govil-landing-page
 - https://docs.openaq.org/
 
**Pollutants of interest:** 
 - for example: pm2.5, pm10, no2
 - "PM" = Particulate Matter:  They are small enough to penetrate deep into the lungs and even enter the bloodstream, potentially leading to respiratory and cardiovascular problems
 - Long-term exposure to high levels of nitrogen dioxide (no2) can cause chronic lung disease.
 - the number is the size of the particles in units of um
 
 
 
 
 
 Israel has not enough data points here!!!!
 
 | Country | City         | AQ on OpenAQ | COVID Cases | GCMR Mobility | Weather | Notes                 |
| ------- | ------------ | ------------ | ----------- | ------------- | ------- | --------------------- |
| 🇩🇪    | **Berlin**   | ✅ Yes        | ✅ RKI       | ✅ Yes         | ✅ Yes   | Best all-around in DE |
| 🇩🇪    | Munich       | ✅ Yes        | ✅           | ✅             | ✅       | Great too             |
| 🇺🇸    | **New York** | ✅ Excellent  | ✅ CDC/NYC   | ✅             | ✅       | Best US choice        |
| 🇺🇸    | Los Angeles  | ✅ Yes        | ✅           | ✅             | ✅       | Strong secondary      |
| 🇺🇸    | Chicago      | ✅ Yes        | ✅           | ✅             | ✅       | Lighter, still viable |


In [21]:
# doesnt exist: Tel Aviv-Yafo! Only Raanana, and Zichon Ya'akov does exist in Israel and data only since Feb 2025
# Germany, India or US has more data points here!!!

import pandas as pd
import requests

def fetch_historical_openaq_by_location(lat=40.7128, lon=-74.0060, parameter='pm25',
                                        start_date='2020-02-15', end_date='2022-02-15'):
    url = 'https://api.openaq.org/v2/measurements'
    headers = {
        'x-api-key': 'bae4d4e1ced71cc0f945ad00bb74cb6122831215042a6757a64aea710727069e'
    }

    # Accumulate results over multiple pages (pagination)
    limit = 10000
    page = 1
    all_results = []

    while True:
        params = {
            'coordinates': f'{lat},{lon}',
            'radius': 10000,
            'parameter': parameter,
            'date_from': start_date,
            'date_to': end_date,
            'limit': limit,
            'page': page,
            'sort': 'asc',
            'order_by': 'datetime'
        }

        response = requests.get(url, headers=headers, params=params)
        data = response.json()

        results = data.get('results', [])
        if not results:
            break

        all_results.extend(results)
        page += 1

        # Safety stop: OpenAQ free tier might not allow fetching all data in one go
        if page > 5:
            print("Page limit reached; consider narrowing the date range.")
            break

    # Convert to DataFrame
    df = pd.DataFrame(all_results)
    if df.empty:
        print("No data found.")
        return df

    df['datetime_utc'] = pd.to_datetime(df['date'].apply(lambda x: x['utc']))
    df['date'] = df['datetime_utc'].dt.date
    df_daily = df.groupby('date')['value'].mean().reset_index()
    df_daily = df_daily.rename(columns={'value': parameter})
    return df_daily

# Example usage
df_pm25_hist = fetch_historical_openaq_by_location()
print(df_pm25_hist.head())


No data found.
Empty DataFrame
Columns: []
Index: []


In [21]:
def list_openaq_cities(country='IL'):
    url = 'https://api.openaq.org/v2/cities'
    params = {
        'country': country,
        'limit': 1000
    }
    headers = {'x-api-key': 'bae4d4e1ced71cc0f945ad00bb74cb6122831215042a6757a64aea710727069e'}
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    results = data.get('results', [])

    # Safely convert to DataFrame
    df = pd.json_normalize(results)

    # Show full structure to debug
    print("Available columns:", df.columns.tolist())

    # Return just city names if available
    if 'city' in df.columns:
        return df[['city']]
    else:
        return df

# Now run it
df_cities = list_openaq_cities()
print(df_cities.head(10))


Available columns: []
Empty DataFrame
Columns: []
Index: []


Alternatives:


1. U.S. EPA Air Quality Data (Reliable Source)
Use the EPA Air Quality System (AQS) data for NYC, which includes PM2.5 and other parameters going back decades.

Download historical data by site or ZIP code:

🔗 https://aqs.epa.gov/aqsweb/airdata/download_files.html

Look for: daily_88101_2020.zip (PM2.5 24-hour avg, 88101 is parameter code)

If you're interested in a specific site or borough, I can help you filter the data accordingly.

2. NY State DEC Air Quality Monitoring
The New York State Department of Environmental Conservation publishes air quality datasets, including PM2.5:

🔗 https://data.ny.gov/browse?category=Environment

3. Google’s Air Quality Dataset (via BigQuery)
Google released a high-resolution air quality dataset (PM2.5, NO2, O3) covering 2015–2021:

🔗 https://registry.opendata.aws/google-airquality/

Usable via Google BigQuery or CSV download



### 4) Weather (from NOAA or ERA5)
 - https://open-meteo.com/en/docs



In [19]:
# # Tel Aviv coordinates
# latitude = 32.0853
# longitude = 34.7818

## New York coordinates
latitude = 40.7128
longitude = -74.0060

# Date range
start_date = '2020-02-15'
end_date = '2022-10-15'

# Request weather data
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': latitude,
    'longitude': longitude,
    'start_date': start_date,
    'end_date': end_date,
    'daily': ['temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean'],
    'timezone': 'auto'
}
weather = requests.get(url, params=params).json()
df_weather = pd.DataFrame({
    'date': weather['daily']['time'],
    'temp_max': weather['daily']['temperature_2m_max'],
    'temp_min': weather['daily']['temperature_2m_min'],
    'humidity': weather['daily']['relative_humidity_2m_mean']
})
df_weather['date'] = pd.to_datetime(df_weather['date']).dt.date
df_weather.head()


,date,temp_max,temp_min,humidity
0,2020-02-15,-1.7,-9.3,53
1,2020-02-16,7.6,-3.2,67
2,2020-02-17,9.7,-1.2,68
3,2020-02-18,9.2,0.4,84
4,2020-02-19,8.0,-0.6,60


### Merge everything

In [23]:
# Ensure all 'date' fields are datetime.date
df_mobility['date'] = df_mobility['date'].dt.date

# df = df_mobility.merge(df_pm25, on='date', how='inner')
df = df_mobility.merge(df_weather, on='date', how='left')
df = df.merge(df_cases, on='date', how='left')

print(df.head())


AttributeError: Can only use .dt accessor with datetimelike values

In [5]:
# Plot a sample

plt.figure(figsize=(12,5))
plt.plot(df_merged['date'], df_merged['mobility_index'], label='Mobility Index', color='blue')
plt.plot(df_merged['date'], df_merged['pm25'], label='PM2.5 (μg/m³)', color='red')
plt.legend()
plt.title('Mobility and PM2.5 in Tel Aviv')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


NameError: name 'df_merged' is not defined

<Figure size 864x360 with 0 Axes>

## Part 2: Feature engineering 
Adding lags

Lag 1 is the previous day -> "lag" = delay

### Rolling window prediction setup
Train over the past 30 days and give a prediction for the 31. day. And then shift by 1 day...


Each sample in X is a 30×5 matrix (30 days × 5 features)

Each y is the COVID case count on the next (31st) day

In [18]:
def create_lagged_dataset(df, feature_cols, target_col='covid_cases', window_size=30):
    """
    Converts a time-series dataframe into samples with lag features for supervised learning.
    """
    X, y = [], []
    for i in range(len(df) - window_size):
        window = df.iloc[i:i+window_size]
        X.append(window[feature_cols].values)
        y.append(df.iloc[i + window_size][target_col])
    
    X = np.array(X)  # shape: (samples, 30, num_features)
    y = np.array(y)  # shape: (samples,)
    return X, y


In [ ]:
feature_cols = ['mobility_index', 'pm25', 'temp_max', 'temp_min', 'humidity']
X, y = create_lagged_dataset(df_merge, feature_cols)

print("X shape:", X.shape)  # e.g., (1000, 30, 5)
print("y shape:", y.shape)  # e.g., (1000,)


### Weather smoothing?

## Part 3: Create Model
LSTM: Long short term memory
 - expects input as (batch_size, sequence_length, num_features)
 - output from LSTM: (batch, 30, hidden_dim)
 - output from last hidden: fc(last_hidden)	(batch,) or (batch, 1) = Scalar prediction of COVID cases (day 31)

In [ ]:
X.shape == (samples, 30, 5)  # 30 = sequence length, 5 = features

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class CovidForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset and dataloader
dataset = CovidForecastDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
import torch.nn as nn

class CovidLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)  # x: (batch, seq_len, input_dim)
        last_hidden = out[:, -1, :]  # Take the last timestep's output
        return self.fc(last_hidden).squeeze()


## Part 4: Training


In [ ]:
model = CovidLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(10):
    for xb, yb in loader:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} loss: {loss.item():.4f}")


Add dropout for regularization

Use learning rate scheduler

Track R² or MAE as additional metrics

Try multi-horizon prediction (predict next 7 days instead of just 1)

